# Hospital Beds Data Warehouse – Cloud Computing Project

**Module:** COMP47780 – Cloud Computing (2025/26 Autumn)  
**Name:** Hemanathan Sasikala Karthikeyan  
**Student ID:** 25201772  

This notebook designs and builds a small data warehouse for a synthetic **Hospital Beds Management** dataset.  
The goal is to model hospital operations so that managers can analyse **bed capacity**, **patient flow** and **staffing levels** over time.

In this notebook I:

- Explore the source tables for patients, services and staff.
- Design a **star schema** with clear fact and dimension tables.
- Enrich the data with useful derived fields (e.g. length of stay, age groups).
- Build the fact and dimension tables in Pandas.
- Export the final tables as **CSV** and **Parquet** files, ready to be loaded into a cloud data platform.

This provides a realistic example of how raw operational data can be transformed into an **analytics-ready warehouse** for cloud-based reporting and dashboards.


In [1]:
import pandas as pd

# Path from notebook folder to raw_data
DATA_PATH = "../raw_data/"

# Read the CSV files
patients = pd.read_csv(DATA_PATH + "patients.csv")
services_weekly = pd.read_csv(DATA_PATH + "services_weekly.csv")
staff = pd.read_csv(DATA_PATH + "staff.csv")
staff_schedule = pd.read_csv(DATA_PATH + "staff_schedule.csv")

print("Loaded all tables successfully!")


Loaded all tables successfully!


In [2]:
print("patients.info():")
patients.info()
print("\nservices_weekly.info():")
services_weekly.info()
print("\nstaff.info():")
staff.info()
print("\nstaff_schedule.info():")
staff_schedule.info()


patients.info():
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 7 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   patient_id      1000 non-null   object
 1   name            1000 non-null   object
 2   age             1000 non-null   int64 
 3   arrival_date    1000 non-null   object
 4   departure_date  1000 non-null   object
 5   service         1000 non-null   object
 6   satisfaction    1000 non-null   int64 
dtypes: int64(2), object(5)
memory usage: 54.8+ KB

services_weekly.info():
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 208 entries, 0 to 207
Data columns (total 10 columns):
 #   Column                Non-Null Count  Dtype 
---  ------                --------------  ----- 
 0   week                  208 non-null    int64 
 1   month                 208 non-null    int64 
 2   service               208 non-null    object
 3   available_beds        208 non-null    int64 
 

In [3]:
print("=== patients ===")
print(patients.describe(include="all"))
print("\nUnique services in patients:", patients["service"].unique())
print("\nMin arrival_date:", patients["arrival_date"].min())
print("Max departure_date:", patients["departure_date"].max())

print("\n\n=== services_weekly ===")
print(services_weekly.describe(include="all"))
print("\nUnique services in services_weekly:", services_weekly["service"].unique())
print("Week range:", services_weekly["week"].min(), "to", services_weekly["week"].max())
print("Months present:", sorted(services_weekly["month"].unique()))

print("\n\n=== staff ===")
print(staff["role"].value_counts())
print("\nUnique services in staff:", staff["service"].unique())

print("\n\n=== staff_schedule ===")
print("Rows:", len(staff_schedule))
print("Weeks:", staff_schedule["week"].min(), "to", staff_schedule["week"].max())
print("Present flag values:", staff_schedule["present"].unique())


=== patients ===
          patient_id                name          age arrival_date  \
count           1000                1000  1000.000000         1000   
unique          1000                 993          NaN          344   
top     PAT-09484753  Samantha Robertson          NaN   2025-01-19   
freq               1                   2          NaN            9   
mean             NaN                 NaN    45.337000          NaN   
std              NaN                 NaN    25.999912          NaN   
min              NaN                 NaN     0.000000          NaN   
25%              NaN                 NaN    23.000000          NaN   
50%              NaN                 NaN    46.000000          NaN   
75%              NaN                 NaN    68.000000          NaN   
max              NaN                 NaN    89.000000          NaN   

       departure_date    service  satisfaction  
count            1000       1000   1000.000000  
unique            337          4           N

## 1. Data sources and tables

For this project, I use the **Hospital Beds Management** dataset from Kaggle.  
It is synthetic but designed to look like realistic hospital data and contains four related tables that will feed into the warehouse design.

- **patients (1,000 rows)**  
  Each row represents a single patient stay. Key fields include:
  - `patient_id`
  - `age`
  - `arrival_date` and `departure_date`
  - `service` (the hospital service where the patient was treated)
  - `satisfaction` (a score representing the patient’s experience)

- **services_weekly (208 rows)**  
  One row per **week–service** combination. For each service and week it records:
  - `available_beds`
  - `patients_request`
  - `patients_admitted`
  - `patients_refused`
  - aggregated `patient_satisfaction`
  - aggregated `staff_morale`
  - an `event` description (e.g. special circumstances affecting capacity)

- **staff (110 rows)**  
  A reference table describing the hospital workforce, with:
  - `staff_id`
  - `staff_name`
  - `role` (e.g. nurse, doctor, admin)
  - `service` (which service the staff member is assigned to)

- **staff_schedule (6,552 rows)**  
  Weekly attendance records that link staff to time and service:
  - `week`
  - `staff_id`
  - `service`
  - `present` (indicator of whether the staff member was present that day/week)

Even though the data are simulated, the tables and relationships closely mirror what you would see in a real hospital information system.  
They provide a good foundation for building a **cloud-ready hospital bed and patient flow analytics warehouse**.


## 2. Data warehouse design – star schema

The goal of this warehouse is to analyse **hospital bed capacity, patient flow and staffing levels over time**.  
To support flexible reporting, I design a **star schema** with **three fact tables** and **four dimension tables**.

### 2.1 Fact tables (measures and grain)

- **fact_service_capacity_weekly**

  - **Grain:** one row per *service per calendar week*  
  - **Source:** `services_weekly`  
  - **Key fields:** `week`, `service`  
  - **Measures and KPIs:**
    - `available_beds`
    - `patients_request`
    - `patients_admitted`
    - `patients_refused`
    - aggregated `patient_satisfaction`
    - aggregated `staff_morale`
    - derived KPIs such as:
      - `utilisation_rate` = `patients_admitted` / `available_beds`
      - `refusal_rate` = `patients_refused` / `patients_request`
  - **Use case:** monitor how well each service matches demand week-by-week and spot pressure points (e.g. high occupancy or high refusal rates).

- **fact_patient_stay**

  - **Grain:** one row per *patient stay*  
  - **Source:** enriched `patients` data  
  - **Key fields:** `patient_id`, `service`, `arrival_date`, `departure_date`  
  - **Measures:**
    - `length_of_stay` (in days)
    - `satisfaction`
  - **Use case:** analyse stay duration and experience by **service**, **age group** and **time period**.

- **fact_staff_attendance**

  - **Grain:** one row per *staff member per week per service*  
  - **Source:** aggregated `staff_schedule`  
  - **Key fields:** `week`, `staff_id`, `service`  
  - **Measure:**
    - `days_present` (total days present in the week)
  - **Use case:** understand staffing patterns and compare attendance against demand and outcomes.

### 2.2 Dimension tables (context)

- **dim_service**  
  - One row per hospital service (e.g. Surgery, ICU, General Medicine).  
  - Used to slice all fact tables by clinical service.

- **dim_week**  
  - One row per calendar week.  
  - Includes the numeric `week` and `month`.  
  - Provides the time dimension for trend and seasonality analysis.

- **dim_patient**  
  - One row per patient.  
  - Columns: `patient_id`, `age`, `age_group`.  
  - Links to `fact_patient_stay` and supports demographic analysis.

- **dim_staff**  
  - One row per staff member.  
  - Columns: `staff_id`, `staff_name`, `role`, `service`.  
  - Links to `fact_staff_attendance` and allows analysis by staff role and assigned service.

By separating **numeric measures** (facts) from **descriptive attributes** (dimensions), this star schema is easy to query and understand.  
It supports typical questions that hospital managers might ask, for example:

- Which services and weeks have the highest bed utilisation and refusal rates?  
- How does patient satisfaction vary by service, age group and length of stay?  
- Are there weeks where low staff attendance coincides with high refusal rates?


## 3.1 Prepare and enrich the base data

In this step I create a working copy of the raw `patients` table called `patients_enriched` and add some useful derived fields:

- Convert `arrival_date` and `departure_date` from strings to proper **datetime** columns.
- Compute a new `length_of_stay` column as the number of days between `arrival_date` and `departure_date`.
- Create an `age_group` band using four brackets: **0–17**, **18–39**, **40–64** and **65+**.

These enriched attributes will be reused in both the **patient dimension** (`dim_patient`) and the **patient stay fact table** (`fact_patient_stay`), so it is important to calculate them once in a clean, consistent way.


In [4]:
# --- 3.1 Convert date columns in patients and create length_of_stay ---

patients_enriched = patients.copy()

# Convert to datetime
patients_enriched["arrival_date"] = pd.to_datetime(patients_enriched["arrival_date"])
patients_enriched["departure_date"] = pd.to_datetime(patients_enriched["departure_date"])

# Length of stay in days
patients_enriched["length_of_stay"] = (
    patients_enriched["departure_date"] - patients_enriched["arrival_date"]
).dt.days

# Create age_group for analysis
bins = [0, 18, 40, 65, 200]
labels = ["0-17", "18-39", "40-64", "65+"]
patients_enriched["age_group"] = pd.cut(patients_enriched["age"], bins=bins, labels=labels, right=False)

patients_enriched.head()


,patient_id,name,age,arrival_date,departure_date,service,satisfaction,length_of_stay,age_group
0,PAT-09484753,Richard Rodriguez,24,2025-03-16,2025-03-22,surgery,61,6,18-39
1,PAT-f0644084,Shannon Walker,6,2025-12-13,2025-12-14,surgery,83,1,0-17
2,PAT-ac6162e4,Julia Torres,24,2025-06-29,2025-07-05,general_medicine,83,6,18-39
3,PAT-3dda2bb5,Crystal Johnson,32,2025-10-12,2025-10-23,emergency,81,11,18-39
4,PAT-08591375,Garrett Lin,25,2025-02-18,2025-02-25,ICU,76,7,18-39


## 3.2 Create dimension tables

Next, I build the four dimension tables that provide descriptive context for the facts.  
Each dimension is created by taking distinct values from the source tables and keeping only the fields needed for analysis.

- **dim_service**

  - Derived from `services_weekly`.
  - Contains a unique list of hospital services.
  - Used as a shared service dimension for all fact tables.

- **dim_week**

  - Derived from `services_weekly`.
  - Contains one row per `week`, along with the corresponding `month`.
  - Acts as the main time dimension for trend and seasonal analysis.

- **dim_patient**

  - Derived from `patients_enriched`.
  - Keeps `patient_id`, `age` and the derived `age_group`.
  - Used to break down metrics in `fact_patient_stay` by patient demographics.

- **dim_staff**

  - Derived from the `staff` reference table.
  - Includes `staff_id`, `staff_name`, `role` and `service`.
  - Links to `fact_staff_attendance` to analyse staffing by role and assigned service.

These dimensions are intentionally **narrow** and **clean** so that they can be joined easily to the fact tables inside a BI or analytics tool.


In [5]:
# --- 3.2.1 dim_service ---

dim_service = (
    services_weekly[["service"]]
    .drop_duplicates()
    .sort_values("service")
    .reset_index(drop=True)
)

dim_service.head()


,service
0,ICU
1,emergency
2,general_medicine
3,surgery


In [6]:
# --- 3.2.2 dim_week ---

dim_week = (
    services_weekly[["week", "month"]]
    .drop_duplicates()
    .sort_values("week")
    .reset_index(drop=True)
)

dim_week.head()


,week,month
0,1,1
1,2,1
2,3,1
3,4,1
4,5,2


In [7]:
# --- 3.2.3 dim_patient ---

dim_patient = (
    patients_enriched[["patient_id", "age", "age_group"]]
    .drop_duplicates()
    .reset_index(drop=True)
)

dim_patient.head()


,patient_id,age,age_group
0,PAT-09484753,24,18-39
1,PAT-f0644084,6,0-17
2,PAT-ac6162e4,24,18-39
3,PAT-3dda2bb5,32,18-39
4,PAT-08591375,25,18-39


In [8]:
# --- 3.2.4 dim_staff ---

dim_staff = (
    staff[["staff_id", "staff_name", "role", "service"]]
    .drop_duplicates()
    .reset_index(drop=True)
)

dim_staff.head()


,staff_id,staff_name,role,service
0,STF-5ca26577,Allison Hill,doctor,emergency
1,STF-02ae59ca,Noah Rhodes,doctor,emergency
2,STF-d8006e7c,Angie Henderson,doctor,emergency
3,STF-212d8b31,Daniel Wagner,doctor,emergency
4,STF-107a58e4,Cristian Santos,doctor,emergency


## 3.3 Create fact tables

With the dimensions in place, I now define the three fact tables that hold the core measures for the warehouse.

- **fact_service_capacity_weekly**

  - Starts from the full `services_weekly` table.
  - Keeps the weekly capacity and demand metrics for each service.
  - Adds two derived KPIs:
    - `utilisation_rate` = `patients_admitted` / `available_beds`
    - `refusal_rate` = `patients_refused` / `patients_request`
  - This table is ideal for dashboards that track bed usage, demand and performance over time.

- **fact_patient_stay**

  - Built from the enriched `patients_enriched` table.
  - Keeps one row per patient stay with:
    - `patient_id`
    - `service`
    - `arrival_date`
    - `departure_date`
    - `length_of_stay`
    - `satisfaction`
  - Supports analysis of how long patients stay and how satisfied they are, by service and demographic group.

- **fact_staff_attendance**

  - Aggregates the `staff_schedule` table.
  - Groups by `week`, `staff_id` and `service`, and sums the `present` flag.
  - Renames the aggregated column to `days_present`.
  - Provides a compact view of how often each staff member was present in a given week and service.

Together, these three fact tables capture **capacity and demand**, **patient experience** and **staff attendance**, giving a rounded view of hospital operations.


In [9]:
# --- 3.3.1 fact_service_capacity_weekly ---

fact_service_capacity_weekly = services_weekly.copy()

# Derived KPIs
fact_service_capacity_weekly["utilisation_rate"] = (
    fact_service_capacity_weekly["patients_admitted"] /
    fact_service_capacity_weekly["available_beds"].replace(0, pd.NA)
)

fact_service_capacity_weekly["refusal_rate"] = (
    fact_service_capacity_weekly["patients_refused"] /
    fact_service_capacity_weekly["patients_request"].replace(0, pd.NA)
)

fact_service_capacity_weekly.head()


,week,month,service,available_beds,patients_request,patients_admitted,patients_refused,patient_satisfaction,staff_morale,event,utilisation_rate,refusal_rate
0,1,1,emergency,32,76,32,44,67,70,none,1.0,0.578947
1,1,1,surgery,45,130,45,85,83,78,flu,1.0,0.653846
2,1,1,general_medicine,37,201,37,164,97,43,flu,1.0,0.815920
3,1,1,ICU,22,31,22,9,84,91,flu,1.0,0.290323
4,2,1,emergency,28,169,28,141,75,64,none,1.0,0.834320


In [10]:
# --- 3.3.2 fact_patient_stay ---

fact_patient_stay = patients_enriched[[
    "patient_id",
    "service",
    "arrival_date",
    "departure_date",
    "length_of_stay",
    "satisfaction"
]].copy()

fact_patient_stay.head()


,patient_id,service,arrival_date,departure_date,length_of_stay,satisfaction
0,PAT-09484753,surgery,2025-03-16,2025-03-22,6,61
1,PAT-f0644084,surgery,2025-12-13,2025-12-14,1,83
2,PAT-ac6162e4,general_medicine,2025-06-29,2025-07-05,6,83
3,PAT-3dda2bb5,emergency,2025-10-12,2025-10-23,11,81
4,PAT-08591375,ICU,2025-02-18,2025-02-25,7,76


In [11]:
# --- 3.3.3 fact_staff_attendance ---

fact_staff_attendance = (
    staff_schedule
    .groupby(["week", "staff_id", "service"], as_index=False)["present"]
    .sum()
    .rename(columns={"present": "days_present"})
)

fact_staff_attendance.head()


,week,staff_id,service,days_present
0,1,STF-038ff4c9,general_medicine,1
1,1,STF-03fbeddc,emergency,1
2,1,STF-052894a3,ICU,0
3,1,STF-05591498,ICU,1
4,1,STF-064febb6,emergency,1


In [12]:
print("dim_service:", dim_service.shape)
print("dim_week:", dim_week.shape)
print("dim_patient:", dim_patient.shape)
print("dim_staff:", dim_staff.shape)

print("\nfact_service_capacity_weekly:", fact_service_capacity_weekly.shape)
print("fact_patient_stay:", fact_patient_stay.shape)
print("fact_staff_attendance:", fact_staff_attendance.shape)


dim_service: (4, 1)
dim_week: (52, 2)
dim_patient: (1000, 3)
dim_staff: (110, 4)

fact_service_capacity_weekly: (208, 12)
fact_patient_stay: (1000, 6)
fact_staff_attendance: (6552, 4)


## 4. Export fact and dimension tables

Finally, I persist all dimension and fact tables to disk so they can be loaded into a cloud data warehouse or BI tool.

- I define an `EXPORT_PATH` pointing to `../exports/`.
- If the folder does not exist, it is created for safety.
- All seven tables are exported:
  - `dim_service`
  - `dim_week`
  - `dim_patient`
  - `dim_staff`
  - `fact_service_capacity_weekly`
  - `fact_patient_stay`
  - `fact_staff_attendance`
- For each table, I write:
  - a **Parquet** file (efficient columnar format, suitable for tools like AWS Athena), and  
  - a **CSV** file (simple text format, easy to inspect and share).

These exported tables form the **cleaned, modelled layer** of the hospital beds analytics warehouse and are ready to be queried or visualised in downstream tools.


In [13]:
EXPORT_PATH = "../exports/"

# If this folder doesn’t exist, create it (safety)
import os
os.makedirs(EXPORT_PATH, exist_ok=True)

tables_to_export = {
    "dim_service": dim_service,
    "dim_week": dim_week,
    "dim_patient": dim_patient,
    "dim_staff": dim_staff,
    "fact_service_capacity_weekly": fact_service_capacity_weekly,
    "fact_patient_stay": fact_patient_stay,
    "fact_staff_attendance": fact_staff_attendance,
}

for name, df in tables_to_export.items():
    parquet_path = os.path.join(EXPORT_PATH, f"{name}.parquet")
    csv_path = os.path.join(EXPORT_PATH, f"{name}.csv")

    # Save as Parquet (for Athena) and CSV (easy to inspect)
    df.to_parquet(parquet_path, index=False)
    df.to_csv(csv_path, index=False)

    print(f"Saved {name} to:")
    print("  ", parquet_path)
    print("  ", csv_path)


Saved dim_service to:
   ../exports/dim_service.parquet
   ../exports/dim_service.csv
Saved dim_week to:
   ../exports/dim_week.parquet
   ../exports/dim_week.csv
Saved dim_patient to:
   ../exports/dim_patient.parquet
   ../exports/dim_patient.csv
Saved dim_staff to:
   ../exports/dim_staff.parquet
   ../exports/dim_staff.csv
Saved fact_service_capacity_weekly to:
   ../exports/fact_service_capacity_weekly.parquet
   ../exports/fact_service_capacity_weekly.csv
Saved fact_patient_stay to:
   ../exports/fact_patient_stay.parquet
   ../exports/fact_patient_stay.csv
Saved fact_staff_attendance to:
   ../exports/fact_staff_attendance.parquet
   ../exports/fact_staff_attendance.csv
